# LLM Model Selection for Java→Kotlin Migration

**Thesis: model selection is a business decision, not a leaderboard lookup.**

This notebook compares four LLMs (Claude Sonnet 4.5, gpt-5-mini, Gemini 2.5 Flash, Qwen3 Coder) on automated Java→Kotlin migration. It measures correctness, idiom quality, speed, and cost across 36 trials (4 models × 3 snippets × 3 trials), then uses a weighted decision matrix to show how the "best" model flips with business priorities.

**The finding:** all four models pass correctness ~100%, but idiom, speed, and cost do *not* track the SciCode benchmark ranking. Quality-first → Claude; volume-first → Qwen.

> ⚠️ The eval loop makes paid API calls. The results of record live in `eval_results_full.csv` and `eval_summary.csv`. Reload from those — do not re-run the eval.

## 1. Setup & Configuration
Load API keys from `.env` and construct the three API clients (Anthropic, OpenAI, OpenRouter — all through the OpenAI-compatible interface).

In [1]:
%pip install python-dotenv openai pandas

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

In [3]:
load_dotenv(override=True)

openai_key = os.getenv("OPENAI_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")
openrouter_key = os.getenv("OPENROUTER_API_KEY")

for name, key in [("OpenAI", openai_key), ("Anthropic", anthropic_key), ("OpenRouter", openrouter_key)]:
    print(f"{name}: {'set, begins ' + key[:6] if key else 'NOT SET'}")

OpenAI: set, begins sk-pro
Anthropic: set, begins sk-ant
OpenRouter: set, begins sk-or-


In [4]:
openai_client     = OpenAI(api_key=openai_key)
anthropic_client  = OpenAI(api_key=anthropic_key,  base_url="https://api.anthropic.com/v1/")
openrouter_client = OpenAI(api_key=openrouter_key, base_url="https://openrouter.ai/api/v1")

## 2. Locked Benchmark Predictions
Predictions recorded *before* running any eval, so the benchmark ranking can't be retrofitted to the results.

In [5]:
# Predictions locked BEFORE running any eval.
# Source: artificialanalysis.ai, recorded 2026-07-18.
#   aa_coding_score = AA SciCode benchmark (only metric with coverage on all 4 models;
#     AA's newer "Coding Index" has data for just claude-sonnet-4-5 and gpt-5-mini).
#   price_per_mtok  = blended 3:1 input:output.
#     Direct-API models use provider pricing; OpenRouter models use OpenRouter pricing.
#     (Qwen: AA quotes Alibaba first-party $3.00 blended, but via OpenRouter it's $0.62.)
# This is our hypothesis. We test it against real results later.
predictions = pd.DataFrame([
    {"model": "claude-sonnet-4-5",       "client": "anthropic",  "aa_coding_score": 45, "price_per_mtok": 6.00, "predicted_rank": 1},
    {"model": "gpt-5-mini",              "client": "openai",     "aa_coding_score": 39, "price_per_mtok": 0.69, "predicted_rank": 2},
    {"model": "google/gemini-2.5-flash", "client": "openrouter", "aa_coding_score": 39, "price_per_mtok": 0.85, "predicted_rank": 3},
    {"model": "qwen/qwen3-coder",        "client": "openrouter", "aa_coding_score": 36, "price_per_mtok": 0.62, "predicted_rank": 4},
])
predictions

,model,client,aa_coding_score,price_per_mtok,predicted_rank
0,claude-sonnet-4-5,anthropic,45,6.00,1
1,gpt-5-mini,openai,39,0.69,2
2,google/gemini-2.5-flash,openrouter,39,0.85,3
3,qwen/qwen3-coder,openrouter,36,0.62,4


## 3. Kotlin Compile & Behavioral-Test Harness
Compile generated Kotlin with `kotlinc`, then glue a behavioral test onto it to verify it actually *works* — not just that it compiles.

In [6]:
import subprocess
import tempfile
import os

def compile_kotlin(code: str, timeout: int = 120) -> tuple[bool, str]:
    """
    Compile Kotlin source. Returns (success, error_text).
      (True,  "")    -> compiled clean
      (False, "...") -> failed, with the compiler's error message
    """
    with tempfile.TemporaryDirectory() as tmpdir:
        kt_path = os.path.join(tmpdir, "Snippet.kt")
        with open(kt_path, "w") as f:
            f.write(code)

        try:
            result = subprocess.run(
                ["kotlinc", kt_path, "-d", os.path.join(tmpdir, "out.jar")],
                capture_output=True,
                text=True,
                timeout=timeout,
            )
        except subprocess.TimeoutExpired:
            return False, f"Compilation timed out after {timeout}s"

        if result.returncode == 0:
            return True, ""
        return False, result.stderr

In [7]:
good_kotlin = '''
fun add(a: Int, b: Int): Int {
    return a + b
}
'''

broken_kotlin = '''
fun add(a: Int, b: Int): Int {
    return a +
}
'''

ok, err = compile_kotlin(good_kotlin)
print("GOOD  ->", ok)

ok, err = compile_kotlin(broken_kotlin)
print("BROKEN->", ok)
print(err[:150])

GOOD  -> True
BROKEN-> False
/var/folders/ct/j63438g95h3614z9372z2vgh0000gn/T/tmppink3lq9/Snippet.kt:3:15: error: syntax error: Expecting an element.
    return a +
              


In [8]:
def strip_code_fences(text: str) -> str:
    """
    Remove markdown code fences from model output.
    Models wrap code like ```kotlin ... ```, and those backticks
    break kotlinc. This returns just the code inside.
    """
    text = text.strip()
    lines = text.split("\n")

    # Drop the first line if it opens a fence (```kotlin, ```kt, or bare ```)
    if lines and lines[0].startswith("```"):
        lines = lines[1:]

    # Drop the last line if it closes a fence (```)
    if lines and lines[-1].startswith("```"):
        lines = lines[:-1]

    return "\n".join(lines).strip()

In [9]:
sample = '''```kotlin
fun add(a: Int, b: Int): Int {
    return a + b
}
```'''

print(strip_code_fences(sample))

fun add(a: Int, b: Int): Int {
    return a + b
}


In [14]:
def run_kotlin_test(kotlin_code: str, test_code: str, timeout: int = 120) -> tuple[bool, str]:
    """
    Glue a test block onto converted Kotlin, compile, run it.
    Returns (passed, detail).
      (True,  "ALL PASS") -> behaves correctly
      (False, "...")      -> compile failed, crashed, or a check failed
    """
    combined = kotlin_code + "\n\n" + test_code

    with tempfile.TemporaryDirectory() as tmpdir:
        kt_path  = os.path.join(tmpdir, "TestSnippet.kt")
        jar_path = os.path.join(tmpdir, "test.jar")
        with open(kt_path, "w") as f:
            f.write(combined)

        try:
            compile_res = subprocess.run(
                ["kotlinc", kt_path, "-include-runtime", "-d", jar_path],
                capture_output=True, text=True, timeout=timeout,
            )
        except subprocess.TimeoutExpired:
            return False, f"Compile timed out after {timeout}s"

        if compile_res.returncode != 0:
            return False, "COMPILE FAILED:\n" + compile_res.stderr

        try:
            run_res = subprocess.run(
                ["java", "-jar", jar_path],
                capture_output=True, text=True, timeout=timeout,
            )
        except subprocess.TimeoutExpired:
            return False, f"Run timed out after {timeout}s"

        output = run_res.stdout.strip()
        if run_res.returncode == 0 and "ALL PASS" in output:
            return True, output
        return False, "RUNTIME FAIL:\n" + output + run_res.stderr

In [15]:
calc_test = '''
fun main() {
    val c = Calculator()
    check(c.add(2, 3) == 5) { "add failed" }
    check(c.multiply(4, 5) == 20) { "multiply failed" }
    println("ALL PASS")
}
'''

passed, detail = run_kotlin_test(kotlin_out, calc_test)
print("Passed ->", passed)
print(detail)

Passed -> True
ALL PASS


## 4. Java→Kotlin Conversion
The conversion prompt plus two generators: `generate_kotlin` (plain) and `generate_kotlin_with_usage` (also captures token counts for real cost).

In [11]:
# System prompt: sets the model's role and rules once.
CONVERT_SYSTEM_PROMPT = """You are an expert Android engineer who converts Java to idiomatic Kotlin.
Respond with ONLY the Kotlin code. No explanations, no markdown, no commentary.
Write natural, idiomatic Kotlin (data classes, null safety, val over var where possible),
not a line-by-line literal translation of the Java.
Preserve the public structure: keep each Java class as a Kotlin class with the same name
and the same public method signatures, so existing call sites still work. Do not convert
classes into top-level functions or singleton objects."""

# Map each model's "client" label to the actual client object.
CLIENTS = {
    "anthropic":  anthropic_client,
    "openai":     openai_client,
    "openrouter": openrouter_client,
}

def generate_kotlin(model: str, client_name: str, java_code: str) -> str:
    """
    Send Java to a model, get idiomatic Kotlin back (fences stripped).
    """
    client = CLIENTS[client_name]
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": CONVERT_SYSTEM_PROMPT},
            {"role": "user",   "content": f"Convert this Java to Kotlin:\n\n{java_code}"},
        ],
    )
    raw = response.choices[0].message.content
    return strip_code_fences(raw)

In [25]:
def generate_kotlin_with_usage(model: str, client_name: str, java_code: str) -> tuple[str, int, int]:
    """
    Convert Java to Kotlin. Returns (kotlin, input_tokens, output_tokens).
    We now capture token usage so we can compute REAL cost per conversion,
    not a blended-sticker estimate.
    """
    client = CLIENTS[client_name]
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": CONVERT_SYSTEM_PROMPT},
            {"role": "user",   "content": f"Convert this Java to Kotlin:\n\n{java_code}"},
        ],
    )
    raw = response.choices[0].message.content
    kotlin = strip_code_fences(raw)

    # The usage field is what the API actually counted. This is ground truth
    # for cost, far better than guessing from character length.
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens

    return kotlin, input_tokens, output_tokens

In [12]:
java_test = """
public class Calculator {
    public int add(int a, int b) {
        return a + b;
    }
    public int multiply(int a, int b) {
        return a * b;
    }
}
"""

kotlin_out = generate_kotlin("gpt-5-mini", "openai", java_test)
print(kotlin_out)

public class Calculator {
    public fun add(a: Int, b: Int): Int = a + b
    public fun multiply(a: Int, b: Int): Int = a * b
}


In [13]:
ok, err = compile_kotlin(kotlin_out)
print("Compiles ->", ok)
if not ok:
    print(err[:200])

Compiles -> True


## 5. The Test Suite (our benchmark)
Each snippet is a (Java, Kotlin-test) pair chosen to probe something Kotlin does differently from Java, so models that merely transliterate syntax get caught.

In [16]:
# The test suite IS our benchmark. Each snippet is a (java, kotlin_test) pair,
# chosen to probe something Kotlin does differently from Java, so weak models
# that just transliterate Java syntax get caught.

test_suite = [
    {
        "name": "calculator",
        "difficulty": "easy",
        "java": """
public class Calculator {
    public int add(int a, int b) { return a + b; }
    public int multiply(int a, int b) { return a * b; }
}
""",
        "test": """
fun main() {
    val c = Calculator()
    check(c.add(2, 3) == 5) { "add failed" }
    check(c.multiply(4, 5) == 20) { "multiply failed" }
    println("ALL PASS")
}
""",
    },
    {
        "name": "user_lookup",
        "difficulty": "medium",
        "java": """
import java.util.HashMap;
import java.util.Map;

public class UserRegistry {
    private Map<Integer, String> users = new HashMap<>();

    public void addUser(int id, String name) {
        users.put(id, name);
    }

    // Returns null if the user is not found.
    public String getName(int id) {
        return users.get(id);
    }

    // Returns the name, or "Unknown" if not found.
    public String getNameOrDefault(int id) {
        String name = users.get(id);
        if (name != null) {
            return name;
        } else {
            return "Unknown";
        }
    }
}
""",
        "test": """
fun main() {
    val r = UserRegistry()
    r.addUser(1, "Alice")
    check(r.getName(1) == "Alice") { "getName existing failed" }
    check(r.getNameOrDefault(1) == "Alice") { "default existing failed" }
    check(r.getNameOrDefault(99) == "Unknown") { "default missing failed" }
    println("ALL PASS")
}
""",
    },
    {
        "name": "order_filter",
        "difficulty": "hard",
        "java": """
import java.util.ArrayList;
import java.util.List;

public class OrderBook {
    public static class Order {
        private String item;
        private int quantity;
        private double price;
        public Order(String item, int quantity, double price) {
            this.item = item; this.quantity = quantity; this.price = price;
        }
        public String getItem() { return item; }
        public int getQuantity() { return quantity; }
        public double getPrice() { return price; }
    }

    // Total value of all orders with quantity above the given threshold.
    public double totalValueAbove(List<Order> orders, int minQuantity) {
        double total = 0.0;
        for (Order o : orders) {
            if (o.getQuantity() > minQuantity) {
                total += o.getPrice() * o.getQuantity();
            }
        }
        return total;
    }
}
""",
        "test": """
fun main() {
    val book = OrderBook()
    val orders = listOf(
        OrderBook.Order("pen", 10, 1.5),
        OrderBook.Order("book", 2, 20.0),
        OrderBook.Order("pencil", 50, 0.5)
    )
    // pen: 10>5 -> 15.0 ; book: 2>5 no ; pencil: 50>5 -> 25.0 ; total 40.0
    val result = book.totalValueAbove(orders, 5)
    check(result == 40.0) { "expected 40.0 but got $result" }
    println("ALL PASS")
}
""",
    },
]

print(f"Test suite: {len(test_suite)} snippets")
for s in test_suite:
    print(f"  {s['difficulty']:6} - {s['name']}")

Test suite: 3 snippets
  easy   - calculator
  medium - user_lookup
  hard   - order_filter


In [17]:
# Validate the two new snippets on one cheap model BEFORE the full loop.
# If these pass, our hand-written tests are sound.

for snippet in test_suite:
    if snippet["name"] == "calculator":
        continue  # already validated earlier

    print(f"\n=== {snippet['difficulty']} : {snippet['name']} ===")
    kotlin = generate_kotlin("gpt-5-mini", "openai", snippet["java"])
    print("--- generated Kotlin ---")
    print(kotlin)
    passed, detail = run_kotlin_test(kotlin, snippet["test"])
    print("--- result ---")
    print("Passed ->", passed)
    if not passed:
        print(detail[:400])


=== medium : user_lookup ===
--- generated Kotlin ---
class UserRegistry {
    private val users = HashMap<Int, String>()

    fun addUser(id: Int, name: String) {
        users[id] = name
    }

    // Returns null if the user is not found.
    fun getName(id: Int): String? = users[id]

    // Returns the name, or "Unknown" if not found.
    fun getNameOrDefault(id: Int): String = users[id] ?: "Unknown"
}
--- result ---
Passed -> True

=== hard : order_filter ===
--- generated Kotlin ---
class OrderBook {
    data class Order(val item: String, val quantity: Int, val price: Double)

    fun totalValueAbove(orders: List<Order>, minQuantity: Int): Double {
        var total = 0.0
        for (o in orders) {
            if (o.quantity > minQuantity) {
                total += o.price * o.quantity
            }
        }
        return total
    }
}
--- result ---
Passed -> True


## 6. Trial & Full-Eval Runners
`run_single_trial` runs one model on one snippet (generate → compile → test → record); `run_full_eval` sweeps every model × snippet × trial.

In [26]:
def run_single_trial(model: str, client_name: str, snippet: dict, trial_num: int) -> dict:
    """
    One full cycle for one model on one snippet.
    Now also records input/output tokens for real cost calculation.
    """
    row = {
        "model": model,
        "snippet": snippet["name"],
        "difficulty": snippet["difficulty"],
        "trial": trial_num,
        "compiled": False,
        "passed": False,
        "gen_seconds": None,
        "input_tokens": None,
        "output_tokens": None,
        "kotlin": "",
        "error": "",
    }

    try:
        start = time.time()
        kotlin, in_tok, out_tok = generate_kotlin_with_usage(model, client_name, snippet["java"])
        row["gen_seconds"] = round(time.time() - start, 2)
        row["input_tokens"] = in_tok
        row["output_tokens"] = out_tok
        row["kotlin"] = kotlin
    except Exception as e:
        row["error"] = f"GENERATION FAILED: {e}"
        return row

    compiled, compile_err = compile_kotlin(kotlin)
    row["compiled"] = compiled
    if not compiled:
        row["error"] = compile_err[:500]
        return row

    passed, detail = run_kotlin_test(kotlin, snippet["test"])
    row["passed"] = passed
    if not passed:
        row["error"] = detail[:500]

    return row

In [27]:
r = run_single_trial("gpt-5-mini", "openai", test_suite[0], trial_num=1)
print("input_tokens:", r["input_tokens"])
print("output_tokens:", r["output_tokens"])
print("passed:", r["passed"])

KeyboardInterrupt: 

In [19]:
def run_full_eval(models_df, suite, n_trials=3) -> pd.DataFrame:
    """
    Run every model across every snippet, n_trials each.
    Prints progress. Returns all rows as a DataFrame.
    """
    rows = []
    total = len(models_df) * len(suite) * n_trials
    done = 0

    for _, m in models_df.iterrows():
        model, client_name = m["model"], m["client"]
        for snippet in suite:
            for t in range(1, n_trials + 1):
                done += 1
                result = run_single_trial(model, client_name, snippet, t)
                rows.append(result)

                # Live progress line so you can see it working.
                status = "OK  " if result["passed"] else (
                         "COMPILE-FAIL" if result["compiled"] else "FAIL")
                if result["error"].startswith("GENERATION"):
                    status = "GEN-FAIL"
                print(f"[{done:2}/{total}] {model:28} {snippet['name']:13} "
                      f"trial {t}  -> {status}")

    return pd.DataFrame(rows)

## 7. Run the Eval → produces the CSVs
> ⚠️ **This section makes paid API calls and writes the source-of-truth CSVs. Do not re-run.**

The Qwen failure inspection at the end is kept intentionally — it's part of the story (idiomatic-but-broken output).

In [28]:
# Smoke test: 1 trial, easy snippet, ALL four models.
# Verifies every slug/client works BEFORE the expensive full run.
smoke = run_full_eval(predictions, test_suite[:1], n_trials=1)
smoke[["model", "compiled", "passed", "gen_seconds", "error"]]

[ 1/4] claude-sonnet-4-5            calculator    trial 1  -> OK  
[ 2/4] gpt-5-mini                   calculator    trial 1  -> OK  
[ 3/4] google/gemini-2.5-flash      calculator    trial 1  -> OK  
[ 4/4] qwen/qwen3-coder             calculator    trial 1  -> OK  


,model,compiled,passed,gen_seconds,error
0,claude-sonnet-4-5,True,True,7.09,
1,gpt-5-mini,True,True,5.94,
2,google/gemini-2.5-flash,True,True,6.62,
3,qwen/qwen3-coder,True,True,1.06,


In [34]:
results = run_full_eval(predictions, test_suite, n_trials=3)
print("\nDone. Rows collected:", len(results))

[ 1/36] claude-sonnet-4-5            calculator    trial 1  -> OK  
[ 2/36] claude-sonnet-4-5            calculator    trial 2  -> OK  
[ 3/36] claude-sonnet-4-5            calculator    trial 3  -> OK  
[ 4/36] claude-sonnet-4-5            user_lookup   trial 1  -> OK  
[ 5/36] claude-sonnet-4-5            user_lookup   trial 2  -> OK  
[ 6/36] claude-sonnet-4-5            user_lookup   trial 3  -> OK  
[ 7/36] claude-sonnet-4-5            order_filter  trial 1  -> OK  
[ 8/36] claude-sonnet-4-5            order_filter  trial 2  -> OK  
[ 9/36] claude-sonnet-4-5            order_filter  trial 3  -> OK  
[10/36] gpt-5-mini                   calculator    trial 1  -> OK  
[11/36] gpt-5-mini                   calculator    trial 2  -> OK  
[12/36] gpt-5-mini                   calculator    trial 3  -> OK  
[13/36] gpt-5-mini                   user_lookup   trial 1  -> OK  
[14/36] gpt-5-mini                   user_lookup   trial 2  -> OK  
[15/36] gpt-5-mini                   user_lookup

In [35]:
fail = results[(results["model"] == "qwen/qwen3-coder") & (results["error"] != "")]
print("Failed trials:", len(fail))
print("\n--- generated kotlin ---")
print(fail.iloc[0]["kotlin"])
print("\n--- error ---")
print(fail.iloc[0]["error"][:400])

Failed trials: 1

--- generated kotlin ---
import java.util.ArrayList
import java.util.List

class OrderBook {
    data class Order(
        val item: String,
        val quantity: Int,
        val price: Double
    )

    // Total value of all orders with quantity above the given threshold.
    fun totalValueAbove(orders: List<Order>, minQuantity: Int): Double {
        var total = 0.0
        for (order in orders) {
            if (order.quantity > minQuantity) {
                total += order.price * order.quantity
            }
        }
        return total
    }
}

--- error ---
COMPILE FAILED:
/var/folders/ct/j63438g95h3614z9372z2vgh0000gn/T/tmp5h0qbj9p/TestSnippet.kt:32:39: error: argument type mismatch: actual type is 'kotlin.collections.List<OrderBook.Order>', but 'java.util.List<OrderBook.Order>' was expected.
    val result = book.totalValueAbove(orders, 5)
                                      ^^^^^^



## 8. Real Cost Model
Actual per-conversion cost from measured token usage and split input/output pricing — not a blended sticker estimate.

In [28]:
# Real input/output prices per million tokens.
# VERIFY each of these against provider / artificialanalysis pricing before trusting.
# These differ from the blended sticker in predictions: this is what you ACTUALLY pay.

pricing = {
    "claude-sonnet-4-5":       {"input": 3.00,  "output": 15.00},
    "gpt-5-mini":              {"input": 0.25,  "output": 2.00},
    "google/gemini-2.5-flash": {"input": 0.30,  "output": 2.50},
    "qwen/qwen3-coder":        {"input": 0.20,  "output": 0.80},
}

In [29]:
def cost_for_row(row):
    """Real cost of one conversion, using actual token counts and split prices."""
    if row["input_tokens"] is None:
        return None
    p = pricing[row["model"]]
    return (row["input_tokens"] / 1_000_000 * p["input"] +
            row["output_tokens"] / 1_000_000 * p["output"])

results["cost_usd"] = results.apply(cost_for_row, axis=1)

# Summarize: correctness, speed, tokens, and REAL cost per model.
summary = results.groupby("model").agg(
    pass_rate=("passed", "mean"),
    avg_seconds=("gen_seconds", "mean"),
    avg_output_tokens=("output_tokens", "mean"),
    avg_cost_usd=("cost_usd", "mean"),
).round(4)
print(summary)

NameError: name 'results' is not defined

In [39]:
g = results[(results["model"] == "gpt-5-mini") & (results["snippet"] == "order_filter")].iloc[0]
print("output_tokens:", g["output_tokens"])
print(g["kotlin"])

output_tokens: 1355
class OrderBook {
    data class Order(val item: String, val quantity: Int, val price: Double)

    fun totalValueAbove(orders: List<Order>, minQuantity: Int): Double =
        orders.sumOf { if (it.quantity > minQuantity) it.price * it.quantity else 0.0 }
}


## 9. Idiom Judge (LLM-as-judge)
Claude scores each conversion 1–5 for Kotlin idiom against a fixed rubric, independent of correctness.

In [30]:
import json

IDIOM_RUBRIC = """You are a senior Kotlin engineer scoring how IDIOMATIC a Java-to-Kotlin conversion is.
Score 1-5 based ONLY on Kotlin idiom, not correctness (assume it compiles and works):

5 = Fully idiomatic. Uses data classes, null safety (?, ?:), val over var, expression bodies,
    functional collection ops (.map/.filter/.sumOf) where natural. No leftover Java patterns.
3 = Mixed. Some Kotlin idiom but also Java-style code (manual loops, var overuse, java.util imports).
1 = Java transliterated into Kotlin syntax. Manual loops, no null safety, no data classes.

Respond with ONLY a JSON object, no other text:
{"score": <1-5>, "reason": "<one short sentence>"}"""

def judge_idiom(kotlin_code: str) -> tuple[int, str]:
    """Score one Kotlin snippet 1-5 for idiom, using Claude as judge."""
    response = anthropic_client.chat.completions.create(
        model="claude-sonnet-4-5",
        messages=[
            {"role": "system", "content": IDIOM_RUBRIC},
            {"role": "user", "content": f"Score this Kotlin:\n\n{kotlin_code}"},
        ],
    )
    raw = response.choices[0].message.content.strip()
    raw = strip_code_fences(raw)  # in case it wraps JSON in ```
    try:
        parsed = json.loads(raw)
        return int(parsed["score"]), parsed["reason"]
    except Exception as e:
        return -1, f"PARSE FAIL: {raw[:100]}"

In [31]:
# Prove the judge on two known-contrasting outputs.
# Qwen's order_filter used a manual loop + java.util imports (should score LOW).
# gpt-5-mini's used data class + .sumOf (should score HIGH).

qwen_code = results[(results["model"] == "qwen/qwen3-coder") &
                    (results["snippet"] == "order_filter")].iloc[0]["kotlin"]
gpt_code = results[(results["model"] == "gpt-5-mini") &
                   (results["snippet"] == "order_filter")].iloc[0]["kotlin"]

score, reason = judge_idiom(qwen_code)
print(f"Qwen      -> {score}  ({reason})")

score, reason = judge_idiom(gpt_code)
print(f"gpt-5-mini-> {score}  ({reason})")

NameError: name 'results' is not defined

In [42]:
# Score idiom for all 36 outputs. ~2-3 min.
scores, reasons = [], []
for _, row in results.iterrows():
    if row["kotlin"]:
        s, r = judge_idiom(row["kotlin"])
    else:
        s, r = -1, "no code"
    scores.append(s)
    reasons.append(r)

results["idiom_score"] = scores
results["idiom_reason"] = reasons

print("Scored:", len(results), "rows")
print("Any parse failures:", (results["idiom_score"] == -1).sum())

Scored: 36 rows
Any parse failures: 0


## 10. Final Summary & Persist to CSV
Aggregate every metric per model and save the results of record.

In [32]:
# The complete picture. Every model, every metric.
final = results.groupby("model").agg(
    pass_rate=("passed", "mean"),
    avg_idiom=("idiom_score", "mean"),
    avg_seconds=("gen_seconds", "mean"),
    avg_cost_usd=("cost_usd", "mean"),
    avg_output_tokens=("output_tokens", "mean"),
).round(4)

# Bring in the benchmark prediction so we can compare prediction vs reality.
final = final.merge(
    predictions.set_index("model")[["aa_coding_score", "predicted_rank"]],
    left_index=True, right_index=True
)

final = final.sort_values("predicted_rank")
print(final)

NameError: name 'results' is not defined

In [33]:
results.to_csv("eval_results_full.csv", index=False)
final.to_csv("eval_summary.csv")
print("Saved: eval_results_full.csv (all 36 trials), eval_summary.csv (per-model)")

NameError: name 'results' is not defined

## 11. Weighted Decision Matrix
Normalize each metric to 0–1, then weight by two contrasting business profiles to show the winner change with priorities.

In [34]:
matrix = final[["pass_rate", "avg_idiom", "avg_seconds", "avg_cost_usd"]].copy()

def normalize(col, higher_is_better=True):
    """Scale a column to 0-1. Best value -> 1, worst -> 0."""
    lo, hi = col.min(), col.max()
    if hi == lo:
        return col * 0 + 1.0  # all equal -> all get full marks
    norm = (col - lo) / (hi - lo)
    return norm if higher_is_better else 1 - norm

# Normalize each metric so higher = better across the board.
norm = pd.DataFrame(index=matrix.index)
norm["correctness"] = normalize(matrix["pass_rate"],   higher_is_better=True)
norm["idiom"]       = normalize(matrix["avg_idiom"],   higher_is_better=True)
norm["speed"]       = normalize(matrix["avg_seconds"], higher_is_better=False)  # lower time = better
norm["cost"]        = normalize(matrix["avg_cost_usd"],higher_is_better=False)  # lower cost = better

print(norm.round(3))

NameError: name 'final' is not defined

In [46]:
# Two businesses, two different value systems.
profiles = {
    "Quality-first\n(code review matters)": {
        "correctness": 0.35, "idiom": 0.45, "speed": 0.05, "cost": 0.15,
    },
    "Volume-first\n(high-throughput migration)": {
        "correctness": 0.30, "idiom": 0.15, "speed": 0.25, "cost": 0.30,
    },
}

for name, w in profiles.items():
    score = (norm["correctness"] * w["correctness"] +
             norm["idiom"]       * w["idiom"] +
             norm["speed"]       * w["speed"] +
             norm["cost"]        * w["cost"])
    print(f"\n=== {name} ===")
    ranked = score.sort_values(ascending=False).round(3)
    print(ranked)
    print(f"WINNER: {ranked.index[0]}")


=== Quality-first
(code review matters) ===
model
claude-sonnet-4-5          0.839
gpt-5-mini                 0.763
qwen/qwen3-coder           0.560
google/gemini-2.5-flash    0.492
dtype: float64
WINNER: claude-sonnet-4-5

=== Volume-first
(high-throughput migration) ===
model
qwen/qwen3-coder           0.670
claude-sonnet-4-5          0.646
google/gemini-2.5-flash    0.605
gpt-5-mini                 0.525
dtype: float64
WINNER: qwen/qwen3-coder


## 12. Interactive UI — reload from CSV + Gradio
Reloads results from the saved CSVs (no eval re-run) and exposes sliders that recompute the recommended model live.

In [50]:
%pip install "gradio==4.44.1" "huggingface_hub==0.25.2" "gradio_client==1.3.0"

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 436 kB 5.0 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 3.1 MB/s eta 0:00:01
     |████████████████████████████████| 306 kB 946 kB/s eta 0:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.8.0
    Uninstalling huggingface-hub-1.8.0:
      Successfully uninstalled huggingface-hub-1.8.0
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [39]:
import pandas as pd

# Reload the saved results from disk (no need to re-run the eval).
results = pd.read_csv("eval_results_full.csv")
final = pd.read_csv("eval_summary.csv", index_col=0)

# Rebuild the per-model summary if needed, and the normalized matrix.
matrix = final[["pass_rate", "avg_idiom", "avg_seconds", "avg_cost_usd"]].copy()

def normalize(col, higher_is_better=True):
    lo, hi = col.min(), col.max()
    if hi == lo:
        return col * 0 + 1.0
    n = (col - lo) / (hi - lo)
    return n if higher_is_better else 1 - n

norm = pd.DataFrame(index=matrix.index)
norm["correctness"] = normalize(matrix["pass_rate"],    higher_is_better=True)
norm["idiom"]       = normalize(matrix["avg_idiom"],    higher_is_better=True)
norm["speed"]       = normalize(matrix["avg_seconds"],  higher_is_better=False)
norm["cost"]        = normalize(matrix["avg_cost_usd"], higher_is_better=False)

print("Reloaded. final and norm rebuilt.")
print(final)

Reloaded. final and norm rebuilt.
                         pass_rate  avg_idiom  avg_seconds  avg_cost_usd  \
model                                                                      
claude-sonnet-4-5           1.0000     4.8889       3.8500        0.0021   
gpt-5-mini                  1.0000     4.7778      11.7267        0.0014   
google/gemini-2.5-flash     1.0000     4.3333      10.3033        0.0003   
qwen/qwen3-coder            0.8889     4.7778       1.6911        0.0001   

                         avg_output_tokens  aa_coding_score  predicted_rank  
model                                                                        
claude-sonnet-4-5                  85.0000               45               1  
gpt-5-mini                        682.3333               39               2  
google/gemini-2.5-flash            79.0000               39               3  
qwen/qwen3-coder                   82.5556               36               4  


In [40]:
import gradio as gr

def recompute(w_correct, w_idiom, w_speed, w_cost):
    """Reweight the normalized metrics and return (ranked table, winner markdown)."""
    total = w_correct + w_idiom + w_speed + w_cost
    if total == 0:
        total = 1
    wc, wi, ws, wk = w_correct/total, w_idiom/total, w_speed/total, w_cost/total

    score = (norm["correctness"] * wc +
             norm["idiom"]       * wi +
             norm["speed"]       * ws +
             norm["cost"]        * wk)

    ranked = score.sort_values(ascending=False).round(3)
    table = ranked.reset_index()
    table.columns = ["Model", "Weighted Score"]

    winner = ranked.index[0]
    winner_text = f"### 🏆 Recommended model: **{winner}**  (score {ranked.iloc[0]})"
    return table, winner_text

findings = final.reset_index()[
    ["model", "pass_rate", "avg_idiom", "avg_seconds", "avg_cost_usd", "aa_coding_score", "predicted_rank"]
].round(4)
findings.columns = ["Model", "Pass Rate", "Idiom (1-5)", "Avg Sec", "Cost $", "Benchmark", "Predicted Rank"]

with gr.Blocks(title="LLM Selection: Java to Kotlin") as demo:
    gr.Markdown("# LLM Model Selection for Java to Kotlin Migration")
    gr.Markdown("Benchmark predicted one ranking. Real eval fractured into tradeoffs. "
                "Drag the weights to see the winner change based on what a business values.")

    gr.Markdown("## The evidence (36 trials, 4 models, 3 snippets)")
    gr.Dataframe(value=findings, interactive=False)

    gr.Markdown("## Your business priorities")
    with gr.Row():
        s_correct = gr.Slider(0, 1, value=0.35, step=0.05, label="Correctness")
        s_idiom   = gr.Slider(0, 1, value=0.45, step=0.05, label="Idiom quality")
        s_speed   = gr.Slider(0, 1, value=0.05, step=0.05, label="Speed")
        s_cost    = gr.Slider(0, 1, value=0.15, step=0.05, label="Cost")

    winner_md = gr.Markdown()
    result_table = gr.Dataframe(interactive=False)

    inputs = [s_correct, s_idiom, s_speed, s_cost]
    for s in inputs:
        s.change(fn=recompute, inputs=inputs, outputs=[result_table, winner_md])

    demo.load(fn=recompute, inputs=inputs, outputs=[result_table, winner_md])

demo.launch(inbrowser=True)

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
